# JC — Holy OG GPU Colab Factory

One-click entry point for `thebigostreets-byte/jc-the-holy-og`.

**Run order:** connect a GPU runtime, run the cells, authorize Google Drive when prompted, and JC stays synchronized with GitHub while persistent data lives in Drive.

The Colab GPU is for Python/AI/asset-processing workloads. The Three.js game still renders in the browser.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
JC_DRIVE = '/content/drive/MyDrive/JC/Holy-OG'
for name in ['assets','checkpoints','outputs','logs']:
    os.makedirs(os.path.join(JC_DRIVE,name), exist_ok=True)
print('JC Drive workspace:', JC_DRIVE)

In [ ]:
# 2. Sync the real JC repository into Drive
import os, subprocess
REPO = 'https://github.com/thebigostreets-byte/jc-the-holy-og.git'
REPO_DIR = os.path.join(JC_DRIVE, 'repo')
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git','clone','--branch','main',REPO,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',REPO_DIR,'checkout','main'], check=True)
    subprocess.run(['git','-C',REPO_DIR,'reset','--hard','origin/main'], check=True)
print('JC commit:', subprocess.check_output(['git','-C',REPO_DIR,'log','-1','--oneline'],text=True).strip())

In [ ]:
# 4. Install JC asset tooling in Colab's Python environment
# Colab's managed Python is used intentionally. A venv on Google Drive can fail
# because Colab's Python may not include the venv/ensurepip components needed to
# bootstrap a new environment on mounted Drive.
import os, subprocess, sys

subprocess.run([sys.executable,'-m','pip','install','-U','pip','setuptools','wheel'], check=True)
subprocess.run([sys.executable,'-m','pip','install','numpy','pillow','trimesh','pygltflib'], check=True)

if os.path.exists(os.path.join(REPO_DIR,'package.json')):
    if os.path.exists(os.path.join(REPO_DIR,'package-lock.json')):
        subprocess.run(['npm','ci'],cwd=REPO_DIR,check=True)
    else:
        subprocess.run(['npm','install'],cwd=REPO_DIR,check=True)

print('JC asset tooling ready in Colab.')


In [ ]:
# 4. Persistent Python asset-tool environment in Drive
import os, subprocess, sys
VENV = os.path.join(JC_DRIVE,'venv')
if not os.path.exists(os.path.join(VENV,'bin','python')):
    subprocess.run([sys.executable,'-m','venv',VENV],check=True)
PIP = os.path.join(VENV,'bin','pip')
subprocess.run([PIP,'install','-U','pip','setuptools','wheel'],check=True)
subprocess.run([PIP,'install','numpy','pillow','trimesh','pygltflib'],check=True)
if os.path.exists(os.path.join(REPO_DIR,'package.json')):
    subprocess.run(['npm','install'],cwd=REPO_DIR,check=True)
print('Persistent JC environment ready.')

In [ ]:
# 5. Verify CUDA from Colab's GPU Python environment
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
else:
    raise RuntimeError('PyTorch cannot see CUDA. Restart the Colab runtime after selecting GPU.')

## 6. JC browser preview

Optional. This starts the current JC Vite build through Colab's browser proxy. The browser/WebGL renderer remains in the browser; CUDA is reserved for cloud-side workloads.

In [ ]:
## Drive layout

`My Drive/JC/Holy-OG/`

- `repo/` — synchronized JC source
- `assets/` — generated assets
- `checkpoints/` — model/checkpoint files
- `outputs/` — generated outputs
- `logs/` — logs

Python packages are installed into Colab's managed runtime rather than a Drive-mounted virtual environment.


## Drive layout

`My Drive/JC/Holy-OG/`

- `repo/` — synchronized JC source
- `assets/` — generated assets
- `checkpoints/` — model/checkpoint files
- `outputs/` — generated outputs
- `logs/` — logs
- `venv/` — persistent Python asset tooling
